[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/03_minimal_ai_systems_overview.ipynb)

# 03. Minimal AI systems overview

앞으로 분해할 부품들이 실제 시스템에서 어떻게 조립되는지 작은 예제로 한 번에 본다. 성능이 아니라 계산 사슬이 실제로 forward/backward/sampling까지 연결되는지를 확인하는 노트북이다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


device: cuda
torch: 2.11.0+cu128


In [2]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 공통 설정
각 예제는 몇 step만 학습한다. 데이터도 synthetic이다.


## A. Tiny GPT


In [3]:
class TinyGPT(nn.Module):
    def __init__(self, vocab=32, d=16, heads=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.norm = nn.RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d, 4 * d),
            nn.SiLU(),
            nn.Linear(4 * d, d),
        )
        self.head = nn.Linear(d, vocab)

    def forward(self, tokens):
        x = self.emb(tokens)
        n = x.size(1)
        mask = torch.triu(torch.ones(n, n, device=x.device, dtype=torch.bool), diagonal=1)

        h = self.norm(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + a
        x = x + self.ff(self.norm(x))
        return self.head(x)

gpt = TinyGPT().to(device)
opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)

tokens = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8]], device=device)

for step in range(3):
    logits = gpt(tokens[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tokens[:, 1:].reshape(-1))

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("GPT step", step, "loss", round(loss.item(), 4))

with torch.no_grad():
    next_id = gpt(tokens[:, :-1])[:, -1].argmax(dim=-1)
    print("next token id:", next_id.item())


GPT step 0 loss 3.8104
GPT step 1 loss 3.6518
GPT step 2 loss 3.4972
next token id: 7


## B. Tiny ViT


In [4]:
class TinyViT(nn.Module):
    def __init__(self, image=16, patch=4, d=16, heads=2, classes=3):
        super().__init__()
        self.patch = patch
        self.proj = nn.Linear(3 * patch * patch, d)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.block = nn.TransformerEncoderLayer(d, heads, 4 * d, batch_first=True)
        self.head = nn.Linear(d, classes)

    def forward(self, x):
        p = self.patch
        patches = F.unfold(x, kernel_size=p, stride=p).transpose(1, 2)
        tokens = self.proj(patches)
        cls = self.cls.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = self.block(tokens)
        return self.head(tokens[:, 0])

vit = TinyViT().to(device)
opt = torch.optim.AdamW(vit.parameters(), lr=3e-3)

images = torch.randn(4, 3, 16, 16, device=device)
labels = torch.tensor([0, 1, 2, 1], device=device)

for step in range(3):
    logits = vit(images)
    loss = F.cross_entropy(logits, labels)

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("ViT step", step, "loss", round(loss.item(), 4))

print("ViT prediction:", vit(images).argmax(dim=-1))


ViT step 0 loss 1.1621
ViT step 1 loss 1.068
ViT step 2 loss 0.9982
ViT prediction: tensor([2, 1, 2, 1], device='cuda:0')


## C. Tiny 2D DiT + Flow


In [5]:
class Tiny2DDiT(nn.Module):
    def __init__(self, channels=2, size=8, patch=2, d=16, heads=2):
        super().__init__()
        self.patch = patch
        self.size = size
        self.in_proj = nn.Linear(channels * patch * patch, d)
        self.time = nn.Sequential(nn.Linear(1, d), nn.SiLU(), nn.Linear(d, d))
        self.block = nn.TransformerEncoderLayer(d, heads, 4 * d, batch_first=True)
        self.out_proj = nn.Linear(d, channels * patch * patch)

    def forward(self, x, t):
        p = self.patch
        tokens = F.unfold(x, kernel_size=p, stride=p).transpose(1, 2)
        h = self.in_proj(tokens) + self.time(t).unsqueeze(1)
        h = self.block(h)
        patches = self.out_proj(h).transpose(1, 2)
        return F.fold(patches, output_size=(self.size, self.size), kernel_size=p, stride=p)

dit2 = Tiny2DDiT().to(device)
opt = torch.optim.AdamW(dit2.parameters(), lr=3e-3)

x0 = torch.randn(2, 2, 8, 8, device=device)
x1 = torch.randn_like(x0)

for step in range(3):
    t = torch.rand(2, 1, device=device)
    tb = t[:, :, None, None]
    xt = (1 - tb) * x0 + tb * x1
    target_v = x1 - x0

    pred_v = dit2(xt, t)
    loss = F.mse_loss(pred_v, target_v)

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("2D flow step", step, "loss", round(loss.item(), 4))

x = x0[:1].clone()
with torch.no_grad():
    for i in range(3):
        t = torch.tensor([[i / 3]], device=device)
        x = x + (1 / 3) * dit2(x, t)
        print("2D sample", i, "mean", round(x.mean().item(), 4))


2D flow step 0 loss 2.2824
2D flow step 1 loss 2.2103
2D flow step 2 loss 2.0987
2D sample 0 mean 0.0926
2D sample 1 mean 0.1072
2D sample 2 mean 0.1359


## D. Tiny 3D DiT + Flow


In [6]:
class Tiny3DDiT(nn.Module):
    def __init__(self, channels=1, size=4, d=16, heads=2):
        super().__init__()
        self.size = size
        self.in_proj = nn.Linear(channels, d)
        self.time = nn.Sequential(nn.Linear(1, d), nn.SiLU(), nn.Linear(d, d))
        self.block = nn.TransformerEncoderLayer(d, heads, 4 * d, batch_first=True)
        self.out_proj = nn.Linear(d, channels)

    def forward(self, x, t):
        b, c, z, y, w = x.shape
        tokens = x.permute(0, 2, 3, 4, 1).reshape(b, z * y * w, c)
        h = self.in_proj(tokens) + self.time(t).unsqueeze(1)
        h = self.block(h)
        out = self.out_proj(h)
        return out.reshape(b, z, y, w, c).permute(0, 4, 1, 2, 3)

dit3 = Tiny3DDiT().to(device)
opt = torch.optim.AdamW(dit3.parameters(), lr=3e-3)

x0 = torch.randn(2, 1, 4, 4, 4, device=device)
x1 = torch.randn_like(x0)

for step in range(3):
    t = torch.rand(2, 1, device=device)
    tb = t[:, :, None, None, None]
    xt = (1 - tb) * x0 + tb * x1
    target_v = x1 - x0

    pred_v = dit3(xt, t)
    loss = F.mse_loss(pred_v, target_v)

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("3D flow step", step, "loss", round(loss.item(), 4))

x = x0[:1].clone()
with torch.no_grad():
    for i in range(3):
        t = torch.tensor([[i / 3]], device=device)
        x = x + (1 / 3) * dit3(x, t)
        print("3D sample", i, "mean", round(x.mean().item(), 4))


3D flow step 0 loss 2.0888
3D flow step 1 loss 1.9726
3D flow step 2 loss 2.0109
3D sample 0 mean -0.1118
3D sample 1 mean -0.1686
3D sample 2 mean -0.2397


## E. Tiny VLA + Flow Policy


In [7]:
class TinyVLAFlowPolicy(nn.Module):
    def __init__(self, d=16, action_dim=4):
        super().__init__()
        self.vision = nn.Linear(6, d)
        self.language = nn.Embedding(16, d)
        self.fuse = nn.TransformerEncoderLayer(d, 2, 4 * d, batch_first=True)
        self.action_in = nn.Linear(action_dim, d)
        self.time = nn.Linear(1, d)
        self.action_out = nn.Linear(d, action_dim)

    def forward(self, vision_vec, language_ids, action_t, t):
        v = self.vision(vision_vec).unsqueeze(1)
        l = self.language(language_ids)
        context = self.fuse(torch.cat([v, l], dim=1)).mean(dim=1)

        h = context + self.action_in(action_t) + self.time(t)
        return self.action_out(F.silu(h))

policy = TinyVLAFlowPolicy().to(device)
opt = torch.optim.AdamW(policy.parameters(), lr=3e-3)

vision = torch.randn(3, 6, device=device)
language = torch.tensor([[1, 2], [3, 4], [5, 6]], device=device)
a0 = torch.randn(3, 4, device=device)
a1 = torch.randn(3, 4, device=device)

for step in range(3):
    t = torch.rand(3, 1, device=device)
    at = (1 - t) * a0 + t * a1
    target_v = a1 - a0

    pred_v = policy(vision, language, at, t)
    loss = F.mse_loss(pred_v, target_v)

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("VLA flow step", step, "loss", round(loss.item(), 4))

action = a0[:1].clone()
with torch.no_grad():
    for i in range(3):
        t = torch.tensor([[i / 3]], device=device)
        action = action + (1 / 3) * policy(vision[:1], language[:1], action, t)
        print("action step", i, action)


VLA flow step 0 loss 2.4984
VLA flow step 1 loss 2.0969
VLA flow step 2 loss 2.2405
action step 0 tensor([[-1.7388, -0.2742,  0.9386,  0.4222]], device='cuda:0')
action step 1 tensor([[-1.8500, -0.3725,  0.9633,  0.4916]], device='cuda:0')
action step 2 tensor([[-2.0009, -0.4928,  0.9788,  0.5739]], device='cuda:0')


## References and provenance

**[3.A] GPT / Transformer**
- 출처: Vaswani et al.; decoder-only Transformer lineage
- 이 노트북에서 가져온 부분: token→attention→logits→autoregressive output

**[3.B] ViT**
- 출처: Dosovitskiy et al., An Image is Worth 16x16 Words
- 이 노트북에서 가져온 부분: patchify→Transformer

**[3.C] DiT + flow**
- 출처: Peebles & Xie, DiT; Lipman et al., Flow Matching
- 이 노트북에서 가져온 부분: 2D tokenization + velocity field

**[3.D] 3D DiT + flow**
- 출처: modern 3D latent/DiT families
- 이 노트북에서 가져온 부분: 3D tokenization + velocity field

**[3.E] VLA flow policy**
- 출처: π0/openpi lineage
- 이 노트북에서 가져온 부분: vision+language conditioning + continuous action flow
